# Geometry-aware CDFMM versus MagTense

This accuracy-focused comparison uses 1,000 deterministic tiles on a 10 x 10 x 10 lattice (one million source-target pairs). `FAST_N` may be set for a small smoke run. Every CDFMM call constructs a true dense direct plan with an explicit target-to-source identity map; finite-geometry source policy intentionally ignores it and retains finite self terms, while identity maps suppress coincident self interactions only for effective point sources.

The tetrahedron target case intentionally selects `TargetModel.POINT`: its physical tetrahedron record is validated and carried through topology, but the field is sampled at the target representative. Exact tetrahedron-tetrahedron near P2P remains unsupported, so that case is compared with the tetrahedron-to-point result rather than with MagTense.

In [ ]:
import os
import numpy as np
import cdfmm
from magtense import magstatics

RUN_FULL = os.environ.get('RUN_FULL', '1').lower() not in {'0', 'false', 'no'}
FAST_N = int(os.environ.get('FAST_N', '0') or 0)
N_TILES = 1000 if RUN_FULL and FAST_N == 0 else (FAST_N or 64)
SIDE = 10.0e-9
SPACING = 3.0 * SIDE
PRISM = cdfmm.RectangularPrism(0.8 * SIDE, 1.1 * SIDE, 0.6 * SIDE)
TETRA_OFFSETS = np.asarray([
    [-0.25, -0.25, -0.25], [0.75, -0.25, -0.25],
    [-0.25, 0.75, -0.25], [-0.25, -0.25, 0.75],
], dtype=np.float64) * (0.8 * SIDE)
TETRA = cdfmm.Tetrahedron(TETRA_OFFSETS)

# The default is exactly (10, 10, 10); FAST_N takes the first deterministic
# points of a sufficiently large centred lattice.
lattice_side = 10 if N_TILES == 1000 else max(1, int(np.ceil(N_TILES ** (1.0 / 3.0))))
grid = np.indices((lattice_side,) * 3, dtype=np.float64).reshape(3, -1).T
grid -= 0.5 * (lattice_side - 1.0)
centres = np.ascontiguousarray(SPACING * grid[:N_TILES])

# A fixed trigonometric state avoids dependence on platform RNG streams.
phase = np.arange(N_TILES, dtype=np.float64)
magnetisations = np.column_stack((
    7.0e5 + 1.5e5 * np.sin(0.17 * phase),
    -3.0e5 + 2.0e5 * np.cos(0.11 * phase),
    4.0e5 * np.sin(0.07 * phase + 0.3),
))
prism_moments = PRISM.volume * magnetisations
tetra_moments = TETRA.volume * magnetisations
identities = np.arange(N_TILES, dtype=np.int32)

def run_dense(source_geometry, target_geometry, moments, source_sizes=(),
              target_sizes=(), source_tetrahedra=(),
              target_tetrahedra=(),
              source_model=cdfmm.SourceModel.EXACT_GEOMETRY,
              target_model=cdfmm.TargetModel.POINT,
              source_positions=None, target_positions=None):
    source_positions = centres if source_positions is None else np.asarray(source_positions)
    target_positions = centres if target_positions is None else np.asarray(target_positions)
    local_identities = np.arange(len(target_positions), dtype=np.int32)
    plan = cdfmm.DenseDirectPlan(
        source_positions, target_positions,
        source_geometry=source_geometry, target_geometry=target_geometry,
        source_sizes=list(source_sizes), target_sizes=list(target_sizes),
        source_tetrahedra=list(source_tetrahedra),
        target_tetrahedra=list(target_tetrahedra),
        source_model=source_model,
        target_model=target_model,
        # The fixed identity map suppresses self interactions only when
        # the effective dense source model is a point dipole.
        target_source_indices=local_identities.tolist(),
        static_precision='float64',
    )
    return np.asarray(plan.evaluate(
        moments, backend=cdfmm.DenseDirectBackend.PORTABLE))

def magtense_prisms(tile_type, obs_size=None):
    tiles = magstatics.Tiles(
        n=N_TILES, tile_type=tile_type,
        size=[PRISM.hx, PRISM.hy, PRISM.hz],
        offset=np.asfortranarray(centres), rot=[0.0, 0.0, 0.0], M_rem=0.0,
    )
    tiles.M = np.asfortranarray(magnetisations)
    points = np.asfortranarray(centres)
    tensor = magstatics.get_demag_tensor(tiles, points, obs_size=obs_size)
    return np.asarray(magstatics.get_H_field(tiles, points, tensor))

def magtense_tetrahedra():
    # MagTense's tetrahedral vertices are supplied as global (n, 3, 4)
    # coordinates.  Zero offsets prevent a second translation.
    global_vertices = centres[:, None, :] + TETRA_OFFSETS[None, :, :]
    vertices = np.asfortranarray(np.transpose(global_vertices, (0, 2, 1)))
    tiles = magstatics.Tiles(n=N_TILES, tile_type=5, vertices=vertices,
                             offset=np.zeros_like(centres), M_rem=0.0)
    tiles.M = np.asfortranarray(magnetisations)
    points = np.asfortranarray(centres)
    tensor = magstatics.get_demag_tensor(tiles, points)
    return np.asarray(magstatics.get_H_field(tiles, points, tensor))

def magtense_single_prism():
    tiles = magstatics.Tiles(n=1, tile_type=2,
                             size=[PRISM.hx, PRISM.hy, PRISM.hz],
                             offset=np.zeros((1, 3)), M_rem=0.0)
    tiles.M = np.asfortranarray(magnetisations[:1])
    points = np.asfortranarray(np.zeros((1, 3)))
    tensor = magstatics.get_demag_tensor(tiles, points)
    return np.asarray(magstatics.get_H_field(tiles, points, tensor))

def magtense_single_tetra():
    vertices = np.asfortranarray(TETRA_OFFSETS.T[None, :, :])
    tiles = magstatics.Tiles(n=1, tile_type=5, vertices=vertices,
                             offset=np.zeros((1, 3)), M_rem=0.0)
    tiles.M = np.asfortranarray(magnetisations[:1])
    points = np.asfortranarray(np.zeros((1, 3)))
    tensor = magstatics.get_demag_tensor(tiles, points)
    return np.asarray(magstatics.get_H_field(tiles, points, tensor))

def errors(actual, reference):
    delta = actual - reference
    absolute = np.linalg.norm(delta, axis=1)
    scale = np.linalg.norm(reference, axis=1)
    relative = absolute / np.maximum(scale, np.finfo(float).eps)
    return {
        'relative_l2': float(np.linalg.norm(delta) / max(np.linalg.norm(reference), np.finfo(float).eps)),
        'max_absolute': float(absolute.max()),
        'max_relative': float(relative.max()),
        'component_rms': np.sqrt(np.mean(delta * delta, axis=0)),
    }

def record(label, actual, reference):
    result = errors(actual, reference)
    result['label'] = label
    return result

print(f'{N_TILES} tiles; {N_TILES * N_TILES:,} source-target pairs; full={RUN_FULL}')

## Case 1 — RectangularPrism → Point

CDFMM uses exact finite prism P2P with point targets. MagTense uses `tile_type=2` and point observations.

In [ ]:
H_prism_point = run_dense(
    cdfmm.SourceGeometry.RECTANGULAR_PRISM, cdfmm.TargetGeometry.POINT,
    prism_moments, source_sizes=[PRISM],
)
H_prism_point_mt = magtense_prisms(tile_type=2)
case_prism_point = record('prism -> point', H_prism_point, H_prism_point_mt)
# A single anisotropic prism at its representative checks the finite self term.
self_cdfmm = run_dense(
    cdfmm.SourceGeometry.RECTANGULAR_PRISM, cdfmm.TargetGeometry.POINT,
    prism_moments[:1], source_sizes=[PRISM],
    source_positions=centres[:1], target_positions=centres[:1],
)
assert np.all(np.isfinite(self_cdfmm))
self_mt = magtense_single_prism()
assert np.allclose(self_cdfmm, self_mt, rtol=5.0e-5, atol=1.0e-12)
print(case_prism_point)

## Case 2 — RectangularPrism → RectangularPrism

CDFMM retains exact target averaging. MagTense uses its averaged-prism `tile_type=8` path with one `obs_size` triplet per target.

In [ ]:
H_prism_prism = run_dense(
    cdfmm.SourceGeometry.RECTANGULAR_PRISM,
    cdfmm.TargetGeometry.RECTANGULAR_PRISM, prism_moments,
    source_sizes=[PRISM], target_sizes=[PRISM],
    target_model=cdfmm.TargetModel.EXACT_GEOMETRY,
)
obs_size = np.asfortranarray(np.tile([PRISM.hx, PRISM.hy, PRISM.hz], (N_TILES, 1)))
H_prism_prism_mt = magtense_prisms(tile_type=8, obs_size=obs_size)
case_prism_prism = record('prism -> prism average', H_prism_prism, H_prism_prism_mt)
assert np.all(np.isfinite(H_prism_prism))
print(case_prism_prism)

## Cases 3–4 — tetrahedral source and physical tetrahedral target

The four vertices in `TETRA` are centroid-relative offsets. Case 3 uses exact tetrahedron source → point target and compares against MagTense `tile_type=5`. Case 4 attaches the same physical tetrahedron records to the targets but selects `TargetModel.POINT`; therefore it intentionally evaluates at target centroids and must agree with Case 3. Selecting `TargetModel.EXACT_GEOMETRY` would request target averaging, and exact tetrahedron-tetrahedron near P2P is currently unsupported.

In [ ]:
H_tetra_point = run_dense(
    cdfmm.SourceGeometry.TETRAHEDRON, cdfmm.TargetGeometry.POINT,
    tetra_moments, source_tetrahedra=[TETRA],
)
H_tetra_point_mt = magtense_tetrahedra()
case_tetra_point = record('tetrahedron -> point', H_tetra_point, H_tetra_point_mt)
assert np.all(np.isfinite(H_tetra_point))

H_tetra_physical_target = run_dense(
    cdfmm.SourceGeometry.TETRAHEDRON, cdfmm.TargetGeometry.TETRAHEDRON,
    tetra_moments, source_tetrahedra=[TETRA], target_tetrahedra=[TETRA],
    target_model=cdfmm.TargetModel.POINT,
)
case_tetra_target_point = record(
    'physical tetra target + point model', H_tetra_physical_target, H_tetra_point
)
assert case_tetra_target_point['max_absolute'] < 1.0e-12 * max(np.linalg.norm(H_tetra_point), 1.0)
# Explicit tetra self field: the centroid is interior, so it is finite.
self_tetra = run_dense(
    cdfmm.SourceGeometry.TETRAHEDRON, cdfmm.TargetGeometry.POINT,
    tetra_moments[:1], source_tetrahedra=[TETRA],
    source_positions=centres[:1], target_positions=centres[:1],
)
assert np.all(np.isfinite(self_tetra))
self_tetra_mt = magtense_single_tetra()
assert np.allclose(self_tetra, self_tetra_mt, rtol=5.0e-5, atol=1.0e-12)
print(case_tetra_point)
print(case_tetra_target_point)

## Accuracy summary

The table is intentionally compact: no repeated timing benchmark is performed.

In [ ]:
rows = [case_prism_point, case_prism_prism, case_tetra_point, case_tetra_target_point]
print(f"{'case':40s} {'relative L2':>14s} {'max abs':>14s} {'max rel':>14s}")
for row in rows:
    print(f"{row['label']:40s} {row['relative_l2']:14.6e} {row['max_absolute']:14.6e} {row['max_relative']:14.6e}")
assert all(np.isfinite(row['relative_l2']) for row in rows)
assert case_prism_point['relative_l2'] < 5.0e-5
assert case_prism_prism['relative_l2'] < 5.0e-5
assert case_tetra_point['relative_l2'] < 5.0e-5
print('PASS: all geometry/model cases satisfy the accuracy smoke thresholds.')